# DataGen Class Testing

This notebook is a small end-to-end smoke test for `DataGen`. It uses the simple ECM requested as `R-[P,R]-[P,R]`, labelled explicitly as `R1-[P2,R3]-[P4,R5]` so AutoEIS can map parameters unambiguously.

Run this notebook in the AutoREC environment. The setup cell searches from the current kernel working directory upward to find the `generate_data_pipline/` directory, so the notebook does not require a hard-coded launch directory.


## Example Command

Other users can run the same workflow from a shell with:

```bash
python - <<'PY'
from pathlib import Path
import sys

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data_gen.py").exists() and (candidate / "ecm_simplification_functions").is_dir():
            return candidate
        nested = candidate / "generate_data_pipline"
        if (nested / "data_gen.py").exists() and (nested / "ecm_simplification_functions").is_dir():
            return nested
    raise FileNotFoundError("Could not find generate_data_pipline from the current working directory or its parents.")

pipeline_dir = find_pipeline_dir()
repo_dir = pipeline_dir.parent
for path in (repo_dir / "src", repo_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from generate_data_pipline.data_gen import DataGen

generator = DataGen(
    random_ecm_circuit="R1-[P2,R3]-[P4,R5]",
    output_dir=pipeline_dir / "data" / "data_gen_class_testing",
    n_random_candidates=500,
    max_selected_curves=20,
    excluded_simplified_ecms=("R1", "R1-C2"),
    verbose=True,
)

balanced_df, batch_infos = generator.generate_data(
    target_per_relabel=1,
    min_batches=1,
    max_batches=5,
    seed_start=2026,
    n_random_candidates=500,
    max_selected_curves=20,
    export=True,
    export_plots=True,
    export_dataprep=True,
)
print(balanced_df[["original_ecm", "relabel_ecm"]].head())
print(batch_infos)
PY
```


In [8]:
from pathlib import Path
import sys

def find_pipeline_dir(start=Path.cwd()):
    """Find the generate_data_pipline directory containing data_gen.py."""
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data_gen.py").exists() and (candidate / "ecm_simplification_functions").is_dir():
            return candidate
        nested = candidate / "generate_data_pipline"
        if (nested / "data_gen.py").exists() and (nested / "ecm_simplification_functions").is_dir():
            return nested
    raise FileNotFoundError(
        "Could not find generate_data_pipline from the current working "
        "directory or its parents. If your kernel starts elsewhere, set "
        "pipeline_dir manually to the directory containing data_gen.py."
    )

pipeline_dir = find_pipeline_dir()
repo_dir = pipeline_dir.parent
for path in (repo_dir / "src", repo_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

try:
    import autoeis  # noqa: F401
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from tqdm.auto import tqdm  # noqa: F401
except ImportError as exc:
    raise ImportError("Run this notebook in the AutoREC environment.") from exc

from generate_data_pipline.data_gen import DataGen


In [9]:
source_ecm = "R1-[P2,R3]-[P4,R5]"
output_dir = pipeline_dir / "data" / "data_gen_class_testing"

generator = DataGen(
    random_ecm_circuit=source_ecm,
    output_dir=output_dir,
    n_random_candidates=500,
    max_selected_curves=20,
    excluded_simplified_ecms=("R1", "R1-C2"),
    fim_refit_max_iters=3,
    fim_refit_min_iters=1,
    fim_refit_max_nfev=100,
    verbose=True,
)

source_ecm


'R1-[P2,R3]-[P4,R5]'

## Generate a Small Dataset

`target_per_relabel=1` keeps this notebook fast while still exercising sampling, filtering, selection, simplify/FIM relabelling, postprocessing, balancing, and CSV export.

In [ ]:
balanced_df, batch_infos = generator.generate_data(
    target_per_relabel=1,
    min_batches=1,
    max_batches=5,
    seed_start=2026,
    n_random_candidates=500,
    max_selected_curves=20,
    export=False,
)

final_df, export_df, csv_path = generator.export_table_csv(
    balanced_df,
    target_per_relabel=1,
)
dataprep_dir = generator.export_dataprep_folder(final_df)
first_eis_csv = sorted(dataprep_dir.rglob("eis_*.csv"))[0]

print(f"CSV saved to: {csv_path}")
print(f"EISDataPrep folder saved to: {dataprep_dir}")
print(f"Example EIS CSV: {first_eis_csv}")
print(f"Generated rows: {len(final_df)}")
display(final_df[["global_position", "original_ecm", "simplified_ecm", "fim_relabel_ecm", "relabel_ecm"]])
display(export_df.head())
display(pd.read_csv(first_eis_csv).head())

## Plot Generated EIS Data

The final ECM label is used as the plot title.

In [ ]:
row = final_df.iloc[0]
Z = np.asarray(generator.simulate_relabel_impedance(row, generator.random_ecm_freq))
title = str(row["relabel_ecm"])

fig, ax = plt.subplots(figsize=(5.8, 5.2))
ax.plot(np.real(Z), -np.imag(Z), marker="o", markersize=3, linewidth=1.6)
ax.set_title(title, fontsize=10, wrap=True)
ax.set_xlabel("Re(Z) / ohm")
ax.set_ylabel("-Im(Z) / ohm")
ax.grid(True, linestyle=":", alpha=0.35)
ax.set_aspect("equal", adjustable="datalim")
fig.tight_layout()

plot_path = output_dir / "data_gen_class_testing_example.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
print(f"Plot saved to: {plot_path}")
plt.show()

## Batch Metadata

In [ ]:
pd.DataFrame(batch_infos)

## Larger Batch Export

This optional cell generates a larger balanced set and exports it in the EISDataPrep-compatible folder structure. Each final ECM gets its own folder, and each EIS curve is written as `eis_N.csv` with `freq`, `Z_real`, and `Z_imag` columns.

In [10]:
batch_generator = DataGen(
    random_ecm_circuit=source_ecm,
    output_dir=pipeline_dir / "data" / "data_gen_batch",
    n_random_candidates=500,
    max_selected_curves=20,
    excluded_simplified_ecms=("R1", "R1-C2"),
    fim_refit_max_iters=10,
    fim_refit_min_iters=1,
    fim_refit_max_nfev=100,
    verbose=True,
)

batch_balanced_df, batch_infos = batch_generator.generate_data(
    target_per_relabel=100,
    min_batches=1,
    max_batches=100,
    seed_start=2026,
    n_random_candidates=5000,
    max_selected_curves=150,
    export=False,
)

batch_final_df = batch_generator.build_final_relabel_df(
    batch_balanced_df,
    target_per_relabel=100,
)
batch_dataprep_dir = batch_generator.export_dataprep_folder(
    batch_final_df,
    output_dir=pipeline_dir / "data" / "data_gen_batch_eisdataprep",
)

batch_files = sorted(batch_dataprep_dir.rglob("eis_*.csv"))
print(f"EISDataPrep folder saved to: {batch_dataprep_dir}")
print(f"CSV files written: {len(batch_files)}")
display(pd.DataFrame(batch_infos))
display(pd.read_csv(batch_files[0]).head())


Simulating random candidates:   0%|          | 0/5000 [00:00<?, ?it/s]

Simplify + FIM relabel selected curves:   0%|          | 0/150 [00:00<?, ?it/s]

Dropping 0 excluded simplified ECMs
Keeping 150 rows after simplified ECM exclusion
Dropping 0 final relabelled ECMs with P-P series
Keeping 150 final relabelled ECMs
Dropping 0 invalid relabelled ECMs
Keeping 150 valid relabelled ECMs
Batch 1: first-batch target groups 3, discovered groups 3
Added this batch:
R1-[P1,R2]             12
R1-[P1,R2]-P2          21
R1-[P1,R2]-[P2,R3]    100
Accumulated target counts:
R1-[P1,R2]             12
R1-[P1,R2]-P2          21
R1-[P1,R2]-[P2,R3]    100


Simulating random candidates:   0%|          | 0/5000 [00:00<?, ?it/s]

Simplify + FIM relabel selected curves:   0%|          | 0/150 [00:00<?, ?it/s]

...

Simulating random candidates:   0%|          | 0/5000 [00:00<?, ?it/s]

Simplify + FIM relabel selected curves:   0%|          | 0/150 [00:00<?, ?it/s]

Dropping 0 excluded simplified ECMs
Keeping 150 rows after simplified ECM exclusion
Dropping 0 final relabelled ECMs with P-P series
Keeping 150 final relabelled ECMs
Dropping 0 invalid relabelled ECMs
Keeping 150 valid relabelled ECMs
Batch 9: first-batch target groups 3, discovered groups 3
Added this batch:
R1-[P1,R2]            13
R1-[P1,R2]-P2          0
R1-[P1,R2]-[P2,R3]     0
Accumulated target counts:
R1-[P1,R2]            100
R1-[P1,R2]-P2         100
R1-[P1,R2]-[P2,R3]    100
Generated relabel counts before final export trimming:
relabel_ecm
R1-[P1,R2]            100
R1-[P1,R2]-P2         100
R1-[P1,R2]-[P2,R3]    100
Generated rows: 300
Dropping 0 excluded simplified ECMs
Keeping 300 rows after simplified ECM exclusion
EISDataPrep folder saved to: /home/gyl19/AutoREC/generate_data_pipline/data/data_gen_batch_eisdataprep
CSV files written: 300


,batch_id,seed,valid_candidate_count,filtered_candidate_count,selected_curve_count,after_final_filters_count,removed_by_high_frequency_filter,removed_by_excluded_simplified_ecm_filter
0,0,2026,5000,3692,150,150,1308,0
1,1,2027,5000,3668,150,150,1332,0
2,2,2028,5000,3683,150,150,1317,0
3,3,2029,5000,3640,150,150,1360,0
4,4,2030,5000,3655,150,150,1345,0
5,5,2031,5000,3669,150,150,1331,0
6,6,2032,5000,3684,150,150,1316,0
7,7,2033,5000,3648,150,150,1352,0
8,8,2034,5000,3614,150,150,1386,0


,freq,Z_real,Z_imag
0,0.010000,228.829694,-0.000505
1,0.012263,228.829694,-0.000620
2,0.015039,228.829694,-0.000760
3,0.018443,228.829694,-0.000932
4,0.022617,228.829694,-0.001143


In [16]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path("/home/gyl19/AutoREC/generate_data_pipline/data/data_gen_batch_eisdataprep")

csv_files = sorted(root.rglob("*.csv"))

print(f"Found {len(csv_files)} samples.")

for i, csv_file in enumerate(csv_files, 1):

    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.strip()

    plt.figure(figsize=(5, 5))

    plt.plot(
        df["Z_real"],
        -df["Z_imag"],
        lw=2,
        color="C0"
    )

    plt.xlabel("Z' (Ω)")
    plt.ylabel("-Z'' (Ω)")
    plt.title(csv_file.stem)
    plt.grid(True)
    plt.axis("equal")
    plt.tight_layout()

    # Save alongside the CSV
    out_file = csv_file.with_suffix(".png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close()

    if i % 20 == 0 or i == len(csv_files):
        print(f"Saved {i}/{len(csv_files)}")

print("Done.")

Found 300 samples.
Saved 20/300
Saved 40/300
Saved 60/300
Saved 80/300
Saved 100/300
Saved 120/300
Saved 140/300
Saved 160/300
Saved 180/300
Saved 200/300
Saved 220/300
Saved 240/300
Saved 260/300
Saved 280/300
Saved 300/300
Done.
